# M8_8.25–M8_8.27 · Gestión de datos, estadística descriptiva y calidad

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_NAME = "m8-herramientas-data-science"
GITHUB_USER = "REPLACE_WITH_YOUR_GITHUB_USERNAME"  # El profesorado lo cambia una vez antes de publicar

def localizar_repo():
    actual = Path.cwd().resolve()
    for candidato in [actual, *actual.parents]:
        if (candidato / "data" / "input").exists():
            return candidato
    if "google.colab" in sys.modules:
        destino = Path("/content") / REPO_NAME
        if not destino.exists():
            url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
            if GITHUB_USER.startswith("REPLACE_"):
                raise RuntimeError("El profesorado debe configurar GITHUB_USER antes de publicar el repositorio.")
            subprocess.run(["git", "clone", "--depth", "1", url, str(destino)], check=True)
        return destino
    raise FileNotFoundError("No se encuentra la raiz del repositorio.")

ROOT = localizar_repo()
DATA = ROOT / "data" / "input"
DB = ROOT / "data" / "database" / "hidrogeologia.sqlite"
OUTPUT = ROOT / "data" / "output"
for carpeta in [OUTPUT / "tables", OUTPUT / "figures", OUTPUT / "logs"]:
    carpeta.mkdir(parents=True, exist_ok=True)
print("Repositorio:", ROOT)

# M8_8.25 · Organización de datos y SQL

Una fila debe representar una unidad de observación definida. La tabla `pozos` contiene entidades relativamente estables; `mediciones`, observaciones repetidas. Una base relacional conecta ambas mediante `id_pozo`. NoSQL incluye modelos documentales, clave–valor, grafos o columnas y responde a otras necesidades.

In [ ]:
import pandas as pd, numpy as np, sqlite3, matplotlib.pyplot as plt
pozos=pd.read_csv(DATA/"pozos.csv")
med=pd.read_csv(DATA/"mediciones.csv",parse_dates=["fecha"])
print(pozos.shape,med.shape); display(pozos.head(),med.head())

## Crear SQLite desde CSV

En la práctica se puede importar información de archivos a una base. Aquí reconstruimos una base docente a partir de los CSV y después consultamos las tablas.

In [ ]:
db_trabajo=OUTPUT/"hidrogeologia_trabajo.sqlite"
with sqlite3.connect(db_trabajo) as con:
    pozos.to_sql("pozos",con,if_exists="replace",index=False)
    med.to_sql("mediciones",con,if_exists="replace",index=False)
print(db_trabajo)

## Consulta

`SELECT` elige campos, `FROM` tabla, `JOIN` combina mediante claves, `WHERE` filtra y `ORDER BY` ordena.

In [ ]:
sql="""SELECT m.id_pozo,m.fecha,p.acuifero,m.conductividad_uScm
FROM mediciones m JOIN pozos p ON m.id_pozo=p.id_pozo
WHERE p.acuifero='Aluvial' AND m.conductividad_uScm>1000
ORDER BY m.conductividad_uScm DESC"""
with sqlite3.connect(db_trabajo) as con:
    consulta=pd.read_sql_query(sql,con)
display(consulta)

# M8_8.26 · Estadística descriptiva

Primero se formula la pregunta. Mínimo, máximo y rango describen extremos; media y mediana, centro; cuantiles e IQR, posición y dispersión central; desviación estándar, dispersión alrededor de la media.

In [ ]:
s=med.conductividad_uScm
q1,q3=s.quantile([.25,.75])
print("n",s.count(),"min",s.min(),"max",s.max(),"media",s.mean(),"mediana",s.median(),"IQR",q3-q1,"std",s.std())

## Distribución y valores extremos

El histograma y el boxplot ayudan a identificar asimetría y observaciones alejadas. Un punto separado no demuestra error.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].hist(s.dropna(),bins=15,edgecolor="black"); axes[0].set(title="Conductividad",xlabel="uS/cm",ylabel="n")
axes[1].boxplot(s.dropna()); axes[1].set(title="Boxplot",ylabel="uS/cm")
plt.tight_layout(); plt.show()

## Comparación entre grupos

Una media global puede mezclar acuíferos diferentes. Los grupos deben tener significado científico. Una diferencia descriptiva no demuestra significación ni causalidad.

In [ ]:
datos=med.merge(pozos,on="id_pozo",validate="many_to_one")
display(datos.groupby("acuifero").conductividad_uScm.agg(n="count",media="mean",mediana="median",std="std"))

# M8_8.27 · Calidad y relaciones

Revisamos ausencias, duplicados, unidades, rangos físicos, plausibilidad, procedencia y límites de detección. Se conserva el original y se crea una copia de análisis.

In [ ]:
revision=pd.Series({"nivel_ausente":med.profundidad_nivel_m.isna().sum(),"ppt_negativa":(med.precipitacion_mm<0).sum(),"ce_muy_alta":(med.conductividad_uScm>3000).sum(),"duplicados":med.duplicated().sum()})
display(revision)

## Corrección no destructiva

Eliminar, corregir o marcar exige justificación. En este ejemplo se eliminan duplicados exactos, se marca precipitación negativa como ausente y se conserva la conductividad alta con una bandera.

In [ ]:
limpio=med.drop_duplicates().copy()
limpio.loc[limpio.precipitacion_mm<0,"precipitacion_mm"]=np.nan
limpio["revisar_conductividad"]=limpio.conductividad_uScm>3000
analisis=limpio.merge(pozos,on="id_pozo",validate="many_to_one")

## Pearson y Spearman, de forma intuitiva

Pearson resume asociación lineal. Spearman utiliza rangos y resume asociación monotónica. Siempre se observa primero el scatter plot.

In [ ]:
x=np.linspace(1,10,30)
lineal=2*x+np.random.default_rng(1).normal(0,2,30)
curva=x**2
sin_patron=np.random.default_rng(2).normal(size=30)
ej=pd.DataFrame({"x":x,"lineal":lineal,"monotona_curva":curva,"sin_patron":sin_patron})
for y in ["lineal","monotona_curva","sin_patron"]:
    print(y,"Pearson",round(ej.x.corr(ej[y],method="pearson"),2),"Spearman",round(ej.x.corr(ej[y],method="spearman"),2))

## Aplicación hidrogeológica y observación influyente

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
for aq,g in analisis.groupby("acuifero"):
    ax.scatter(g.profundidad_nivel_m,g.conductividad_uScm,label=aq,alpha=.7)
ax.set(xlabel="Profundidad (m)",ylabel="Conductividad (uS/cm)"); ax.legend(); plt.show()
print(analisis[["profundidad_nivel_m","conductividad_uScm"]].corr(method="pearson"))

## Correlación no implica causalidad

Una asociación entre profundidad y conductividad puede reflejar acuífero, litología, residencia, localización, periodo o un valor influyente. La causalidad requiere un mecanismo físico defendible y evidencia adicional.

## Relación global frente a relaciones dentro de grupos

Una correlación global puede deberse a diferencias entre acuíferos. Por eso se compara el patrón total con el patrón dentro de cada grupo.

In [ ]:
print("Global Spearman",analisis.profundidad_nivel_m.corr(analisis.conductividad_uScm,method="spearman"))
for aq,g in analisis.groupby("acuifero"):
    print(aq,round(g.profundidad_nivel_m.corr(g.conductividad_uScm,method="spearman"),2))

## Dependencia espacial y temporal

Varias mediciones del mismo pozo no representan observaciones independientes. Comparten localización y están próximas en el tiempo. En ML, dividir filas aleatoriamente puede colocar el mismo pozo en entrenamiento y prueba.

In [ ]:
display(analisis.groupby("id_pozo").size().rename("mediciones"))
print("Filas",len(analisis),"pozos",analisis.id_pozo.nunique())

## Actividad

1. Modifica la consulta SQL.
2. Compara media y mediana por acuífero.
3. Identifica dos problemas de calidad y justifica el tratamiento.
4. Calcula Pearson y Spearman globales y por grupo.
5. Explica por qué una asociación observada no demuestra causalidad.
6. Indica una implicación de la dependencia espacial o temporal.